# 1단계: 모델 백본 비교 실험

**목적**: 동일 조건에서 3종 모델의 맥락 이해 능력 비교

| 실험 | 모델 | HuggingFace ID |
|---|---|---|
| M1-A | KLUE-RoBERTa-base | `klue/roberta-base` |
| M1-B | KcELECTRA-base | `beomi/KcELECTRA-base` |
| M1-C | KR-ELECTRA | `snunlp/KR-ELECTRA-discriminator` |

**고정 조건**: Mean Pooling, 차등 LR(4e-6/2e-5), MAX_LEN=256, Dropout=0.3

**실행 방식**: 3종 모델을 자동으로 순회하며 학습 → 평가 → Test 추론

**참조**: `docs/model_comparison_plan.md`

In [ ]:
# ============================================================
# Cell 1: 환경 설정
# ============================================================
import pandas as pd
import numpy as np
import re
import os
import time
import gc
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print('MPS (Apple Silicon)')
else:
    device = torch.device('cpu')
    print('CPU')
print(f'Device: {device}')

In [ ]:
# ============================================================
# Cell 2: 데이터 로드 및 전처리 (1회만 실행)
# ============================================================
train_df = pd.read_csv('../data/train_final.csv')
val_df = pd.read_csv('../data/val_final.csv')
test_df_raw = pd.read_csv('../data/test.csv')

label2id = {
    '갈취 대화': 0, '기타 괴롭힘 대화': 1,
    '직장 내 괴롭힘 대화': 2, '협박 대화': 3, '일반 대화': 4,
}
id2label = {v: k for k, v in label2id.items()}

def preprocess(text):
    if not isinstance(text, str): return ''
    text = text.replace('\n', ' ')
    return re.sub(r'\s+', ' ', text).strip()

train_df['text'] = train_df['conversation'].apply(preprocess)
train_df['label'] = train_df['class'].map(label2id)
val_df['text'] = val_df['conversation'].apply(preprocess)
val_df['label'] = val_df['class'].map(label2id)
test_df_raw['text'] = test_df_raw['conversation'].apply(
    lambda x: preprocess(str(x).replace('"', ''))
)

train_texts = train_df['text'].tolist()
train_labels = train_df['label'].tolist()
val_texts = val_df['text'].tolist()
val_labels = val_df['label'].tolist()
test_texts = test_df_raw['text'].tolist()

print(f'Train: {len(train_texts)}건 | Val: {len(val_texts)}건 | Test: {len(test_texts)}건')
print(train_df['class'].value_counts())

In [ ]:
# ============================================================
# Cell 3: 고정 하이퍼파라미터 + 모델 리스트
# ============================================================

MODELS = [
    ('M1-A', 'klue/roberta-base'),
    ('M1-B', 'beomi/KcELECTRA-base'),
    ('M1-C', 'snunlp/KR-ELECTRA-discriminator'),
]

# 3종 모두 동일하게 고정
MAX_LEN = 256
BATCH_SIZE = 32       # 8 → 32 (T4 GPU 16GB 기준 최적)
LEARNING_RATE = 2e-5
BACKBONE_LR = 4e-6
EPOCHS = 3            # 5 → 3 (B04 best가 epoch 2였으므로 충분)
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
DROPOUT_RATE = 0.3

print('비교 대상 모델:')
for exp_id, name in MODELS:
    print(f'  {exp_id}: {name}')
print(f'\n고정 조건: MAX_LEN={MAX_LEN}, BS={BATCH_SIZE}, HeadLR={LEARNING_RATE}, BackboneLR={BACKBONE_LR}, Epochs={EPOCHS}')

In [ ]:
# ============================================================
# Cell 4: 공통 클래스 및 함수 정의
# ============================================================

class ConversationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
        }


class ConversationClassifier(nn.Module):
    def __init__(self, model_name, num_classes=5, dropout_rate=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        h = self.backbone.config.hidden_size
        self.layer_norm = nn.LayerNorm(h)
        self.fc1 = nn.Linear(h, 256)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(256, num_classes)

    def mean_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        return (last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.layer_norm(self.mean_pooling(out.last_hidden_state, attention_mask))
        return self.fc2(self.dropout(self.activation(self.fc1(pooled))))


def train_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    preds, trues = [], []
    for batch in loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds.extend(logits.argmax(-1).cpu().numpy())
        trues.extend(labels.cpu().numpy())
    return total_loss / len(loader), f1_score(trues, preds, average='macro')


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            logits = model(ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            preds.extend(logits.argmax(-1).cpu().numpy())
            trues.extend(labels.cpu().numpy())
    return total_loss / len(loader), f1_score(trues, preds, average='macro'), preds, trues


print('Dataset, Model, 학습/검증 함수 정의 완료')

In [ ]:
# ============================================================
# Cell 5: 3종 모델 자동 순회 학습
# ============================================================

all_results = []  # 전체 비교 결과 저장

for exp_id, model_name in MODELS:
    model_short = model_name.split('/')[-1]
    print(f'\n{"="*70}')
    print(f'[{exp_id}] {model_name}')
    print(f'{"="*70}')
    
    # --- 시드 재고정 (공정 비교) ---
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    
    # --- 토크나이저 + DataLoader ---
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_dataset = ConversationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
    val_dataset = ConversationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
    test_dataset = ConversationDataset(test_texts, [0]*len(test_texts), tokenizer, MAX_LEN)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # --- 모델 생성 ---
    model = ConversationClassifier(model_name, len(label2id), DROPOUT_RATE).to(device)
    
    # --- 차등 학습률 옵티마이저 ---
    head_params, backbone_params = [], []
    for name, param in model.named_parameters():
        if any(k in name for k in ['fc1', 'fc2', 'layer_norm']):
            head_params.append(param)
        else:
            backbone_params.append(param)
    
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': BACKBONE_LR, 'weight_decay': WEIGHT_DECAY},
        {'params': head_params, 'lr': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY},
    ])
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
    criterion = nn.CrossEntropyLoss()
    
    # --- 학습 ---
    best_f1 = 0
    best_epoch = 0
    history = []
    save_path = f'best_model_{model_short}.pt'
    total_start = time.time()
    
    for epoch in range(EPOCHS):
        start = time.time()
        train_loss, train_f1 = train_epoch(model, train_loader, criterion, optimizer, scheduler, device)
        val_loss, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
        elapsed = time.time() - start
        
        history.append({'epoch': epoch+1, 'train_loss': train_loss, 'train_f1': train_f1,
                        'val_loss': val_loss, 'val_f1': val_f1})
        
        marker = ''
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch + 1
            torch.save(model.state_dict(), save_path)
            marker = ' << BEST'
        
        print(f'  Epoch {epoch+1} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | {elapsed:.0f}s{marker}')
    
    total_time = (time.time() - total_start) / 60
    
    # --- Best Model로 상세 평가 ---
    model.load_state_dict(torch.load(save_path, weights_only=True))
    _, final_f1, final_preds, final_trues = evaluate(model, val_loader, criterion, device)
    
    class_names = [id2label[i] for i in range(5)]
    report = classification_report(final_trues, final_preds, target_names=class_names, digits=4, output_dict=True)
    normal_f1 = report['일반 대화']['f1-score']
    
    print(f'\n  >> Best Epoch: {best_epoch} | Val F1: {best_f1:.4f} | 일반대화 F1: {normal_f1:.4f} | 총 {total_time:.1f}분')
    print(classification_report(final_trues, final_preds, target_names=class_names, digits=4))
    
    # --- Test 추론 ---
    model.eval()
    test_preds = []
    with torch.no_grad():
        for batch in test_loader:
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            test_preds.extend(logits.argmax(-1).cpu().numpy())
    
    test_classes = [str(id2label[p]) for p in test_preds]
    test_normal_count = sum(1 for c in test_classes if c == '일반 대화')
    
    # 제출 파일 저장
    sub_df = test_df_raw[['idx']].copy()
    sub_df['class'] = test_classes
    sub_df.to_csv(f'submission_{model_short}.csv', index=False)
    
    print(f'  >> Test 일반대화: {test_normal_count}건 / 100건 기대')
    print(f'  >> Test 분포: {pd.Series(test_classes).value_counts().to_dict()}')
    
    # --- 결과 저장 ---
    all_results.append({
        'exp_id': exp_id,
        'model': model_short,
        'val_f1': round(best_f1, 4),
        'normal_val_f1': round(normal_f1, 4),
        'test_normal': test_normal_count,
        'best_epoch': best_epoch,
        'time_min': round(total_time, 1),
        'history': history,
    })
    
    # --- 메모리 정리 ---
    del model, optimizer, scheduler, tokenizer
    del train_dataset, val_dataset, test_dataset
    del train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'\n{"="*70}')
print('3종 모델 비교 완료!')
print(f'{"="*70}')

In [ ]:
# ============================================================
# Cell 6: 비교 결과 요약
# ============================================================

comparison = pd.DataFrame([{k: v for k, v in r.items() if k != 'history'} for r in all_results])

print('\n' + '=' * 70)
print('1단계 모델 비교 결과')
print('=' * 70)
print(comparison.to_string(index=False))

# 최적 모델 선정
best_idx = comparison['val_f1'].idxmax()
best = comparison.loc[best_idx]
print(f'\n>> 최적 모델: {best["model"]} ({best["exp_id"]})')
print(f'   Val F1: {best["val_f1"]:.4f} | 일반대화 Val F1: {best["normal_val_f1"]:.4f} | Test 일반대화: {best["test_normal"]}건')
print(f'\n>> 2단계 실험에서 {best["model"]}로 학습 전략 비교 진행')

# CSV 저장
comparison.to_csv('model_comparison_results.csv', index=False)
print('\n결과 저장: model_comparison_results.csv')

In [ ]:
# ============================================================
# Cell 7: 학습 곡선 비교 (3종 한 그래프에)
# ============================================================

colors = ['#3b82f6', '#ef4444', '#10b981']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, result in enumerate(all_results):
    hist = pd.DataFrame(result['history'])
    label = result['model']
    axes[0].plot(hist['epoch'], hist['val_loss'], 'o-', label=label, color=colors[i])
    axes[1].plot(hist['epoch'], hist['val_f1'], 'o-', label=label, color=colors[i])

axes[0].set_title('Val Loss 비교', fontsize=14)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Val F1 비교', fontsize=14)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 Macro')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('1단계: 모델 백본 비교', fontsize=16)
plt.tight_layout()
plt.savefig('model_comparison_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Cell 8: 혼동 행렬 비교 (3종 나란히)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
class_names = [id2label[i] for i in range(5)]

for i, result in enumerate(all_results):
    # Best model 다시 로드해서 val 예측
    model_name = [m for _, m in MODELS if m.split('/')[-1] == result['model']][0]
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    val_dataset = ConversationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = ConversationClassifier(model_name, len(label2id), DROPOUT_RATE).to(device)
    model.load_state_dict(torch.load(f'best_model_{result["model"]}.pt', weights_only=True))
    _, _, preds, trues = evaluate(model, val_loader, nn.CrossEntropyLoss(), device)
    
    cm = confusion_matrix(trues, preds)
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names,
                cmap='Blues', ax=axes[i])
    axes[i].set_title(f'{result["exp_id"]}: {result["model"]}\nVal F1={result["val_f1"]:.4f}', fontsize=11)
    axes[i].set_ylabel('Actual' if i == 0 else '')
    axes[i].set_xlabel('Predicted')
    axes[i].tick_params(axis='x', rotation=30)
    
    del model, tokenizer, val_dataset, val_loader
    gc.collect()

plt.suptitle('혼동 행렬 비교 (3종)', fontsize=15)
plt.tight_layout()
plt.savefig('model_comparison_cm.png', dpi=150, bbox_inches='tight')
plt.show()